# Global History CMA — April 2026
Run all cells top-to-bottom.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

C = dict(
    red    = '#E84855',
    orange = '#F4A261',
    teal   = '#44BBA4',
    green  = '#06D6A0',
    blue   = '#3A86FF',
    navy   = '#2D3142',
    gray   = '#888888',
    lgray  = '#D8D8D8',
)

CLASS_COLORS = {'Bermejo': C['blue'], 'Dushin': C['orange'], 'Kovelsky': C['teal']}

NQ = 9  # placeholder — overwritten when data loads

def score_color(s):
    p = s / NQ
    if p >= 0.78: return C['green']
    if p >= 0.56: return C['teal']
    if p >= 0.34: return C['orange']
    return C['red']

def pct_color(p):
    if p >= 0.70: return C['teal']
    if p >= 0.50: return C['orange']
    return C['red']

PLOTLY_BASE = dict(
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='sans-serif', size=12, color=C['navy']),
    hoverlabel=dict(bgcolor='white', font_size=12),
    margin=dict(l=20, r=20, t=60, b=20),
)

plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
    'axes.titlepad':     10,
    'axes.labelsize':    11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.grid':         True,
    'grid.alpha':        0.35,
    'grid.color':        '#CCCCCC',
    'figure.dpi':        120,
    'legend.framealpha': 0.95,
    'legend.edgecolor':  '#CCCCCC',
    'legend.fontsize':   9.5,
})

In [ ]:
FILE = './Global April CMA Data.xlsx'

scores_raw = pd.read_excel(FILE, sheet_name='Scores')
questions  = pd.read_excel(FILE, sheet_name='Questions')
std_sheet  = pd.read_excel(FILE, sheet_name='Content Standards')

NQ = len(questions)   # auto-detected question count

# Content Standards lookup — rename first two cols regardless of sheet layout
_cols = std_sheet.columns.tolist()
_cols[0], _cols[1] = 'Code', 'Framework'
std_sheet.columns = _cols
std_sheet['Code'] = std_sheet['Code'].astype(str).str.strip()
std_lookup = dict(zip(std_sheet['Code'], std_sheet['Framework'].astype(str)))

def std_name(code, max_len=40):
    c = str(code).strip()
    if not c or c in ('nan', 'None', 'NaN', ''):
        return ''
    name = std_lookup.get(c, '')
    if not name or name.strip() in ('nan', 'None', 'NaN', ''):
        return ''
    return (name[:max_len] + '\u2026') if len(name) > max_len else name

answer_cols  = [f'Q{i}_ans' for i in range(1, NQ + 1)]
correct_cols = [f'Q{i}'     for i in range(1, NQ + 1)]

rename = {scores_raw.columns[k]: v for k, v in enumerate(['Class','Grade','Period','Student','ELL','IEP'])}
for i, c in enumerate(answer_cols):  rename[scores_raw.columns[10 + i]] = c
for i, c in enumerate(correct_cols): rename[scores_raw.columns[10 + NQ + i]] = c

df = scores_raw.rename(columns=rename).dropna(subset=['Class']).copy()
df[correct_cols] = df[correct_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
df['MC_Score'] = df[correct_cols].sum(axis=1)
df['MC_Pct']   = df['MC_Score'] / NQ
df['MC_Blank'] = df[answer_cols].isna().sum(axis=1)
df['Is_ELL']   = df['ELL'].notna()
df['Is_IEP']   = df['IEP'].notna()

q = questions.copy()
q = q.rename(columns={
    'CMA #': 'Q_Num', 'Source #': 'Source_Num', 'Full Answer': 'Full_Answer',
    'Stimulus Type': 'Stimulus_Type', 'Source Type': 'Source_Type',
    'Task Model': 'Task_Model', 'Content': 'Content_Std',
    'Need OI?': 'Need_OI', 'Point Biserial': 'Point_Biserial',
    'Option 1': 'Opt1', 'Option 2': 'Opt2', 'Option 3': 'Opt3', 'Option 4': 'Opt4',
})
q = q.sort_values('Q_Num').reset_index(drop=True)
q['Q_Label'] = q.apply(
    lambda r: (f"Q{int(r['Q_Num'])} \u2014 "
               + str(r['Question']).strip()[:52]
               + ('\u2026' if len(str(r['Question']).strip()) > 52 else '')),
    axis=1
)

print(f"Loaded {len(df)} students  |  {df['Class'].nunique()} classes  |  {NQ} questions  |  "
      f"ELL: {df['Is_ELL'].sum()}  |  IEP: {df['Is_IEP'].sum()}  |  "
      f"Both: {(df['Is_ELL'] & df['Is_IEP']).sum()}")

## 1.  At a Glance

In [ ]:
mean_score = df['MC_Score'].mean()
thresh     = round(NQ * 0.67)
pct_thr    = (df['MC_Score'] >= thresh).mean()
n_students = len(df)
n_blanks   = (df['MC_Blank'] > 0).sum()

kpis = [
    (f"{mean_score:.1f} / {NQ}", "Class Average Score", score_color(round(mean_score))),
    (f"{pct_thr:.0%}", f"Scored {thresh}+ (67%+)",
     C['teal'] if pct_thr >= 0.55 else C['orange'] if pct_thr >= 0.35 else C['red']),
    (f"{n_students}", "Total Students", C['blue']),
    (f"{n_blanks}", "Left a Question Blank",
     C['orange'] if n_blanks > n_students * 0.10 else C['teal']),
]

fig = plt.figure(figsize=(14, 8))
gs  = GridSpec(2, 4, figure=fig, height_ratios=[1.1, 2.4],
               hspace=0.55, wspace=0.2, top=0.93, bottom=0.07, left=0.04, right=0.97)

for i, (val, lbl, color) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(color)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, 0.62, val, ha='center', va='center', transform=ax.transAxes,
            fontsize=26, fontweight='bold', color='white')
    ax.text(0.5, 0.21, lbl, ha='center', va='center', transform=ax.transAxes,
            fontsize=9.5, color='white', alpha=0.93, multialignment='center')

ax_h = fig.add_subplot(gs[1, :])
max_cnt = max((df['MC_Score'] == s).sum() for s in range(0, NQ + 1))
def _bc(s):
    if s > b3: return C['green']
    if s > b2: return C['teal']
    if s > b1: return C['orange']
    return C['red']

for score in range(0, NQ + 1):
    cnt = (df['MC_Score'] == score).sum()
    ax_h.bar(score, cnt, color=_bc(score), width=0.72, edgecolor='white', linewidth=1.5)
    if cnt > 0:
        pass  # no bar labels

ax_h.axvline(mean_score, color='grey', linestyle=':', linewidth=1.5, alpha=0.5, zorder=6)
ax_h.set_xticks(range(0, NQ + 1))
ax_h.set_xlabel(f'Number of Questions Correct  (out of {NQ})', labelpad=8)
ax_h.set_ylabel('Number of Students')
ax_h.set_title('How Did Students Score?')
ax_h.grid(axis='y'); ax_h.grid(axis='x', alpha=0)
ax_h.spines['left'].set_linewidth(0.8); ax_h.spines['bottom'].set_linewidth(0.8)

# Score band legend
b1 = round(NQ * 0.34); b2 = round(NQ * 0.56); b3 = round(NQ * 0.78)
ax_h.legend(handles=[
    mpatches.Patch(color=C['red'],    label=f'0\u2013{b1}   Needs Support'),
    mpatches.Patch(color=C['orange'], label=f'{b1+1}\u2013{b2}  Approaching'),
    mpatches.Patch(color=C['teal'],   label=f'{b2+1}\u2013{b3}  Meeting Standard'),
    mpatches.Patch(color=C['green'],  label=f'{b3+1}\u2013{NQ}  Exceeding'),
], loc='upper left')
plt.suptitle('Global History CMA — April 2026', fontsize=17, fontweight='bold')
plt.show()

## 2.  Class & Period Performance

In [ ]:
cp       = df.groupby(['Class','Period'])['MC_Score'].agg(['mean','count']).reset_index()
classes  = sorted(df['Class'].unique())
bar_width = 0.22

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

ax = axes[0]
group_gap = 0.15
current_x = 0.0
class_centers = {}
for cls in classes:
    cls_data = cp[cp['Class'] == cls].sort_values('Period')
    n_bars   = len(cls_data)
    color    = CLASS_COLORS.get(cls, C['blue'])
    for pi, (_, row) in enumerate(cls_data.iterrows()):
        x_pos = current_x + pi * bar_width
        ax.bar(x_pos, row['mean'], width=bar_width * 0.85, color=color,
               edgecolor='white', linewidth=1.0)
        ax.text(x_pos, row['mean'] - 0.25, f"P{int(row['Period'])}",
                ha='center', va='top', fontsize=10, fontweight='bold', color='white')
    class_centers[cls] = current_x + (n_bars - 1) * bar_width / 2
    current_x += n_bars * bar_width + group_gap

ax.set_xticks(list(class_centers.values()))
ax.set_xticklabels(list(class_centers.keys()), fontsize=12, fontweight='bold')
ax.set_xlim(-0.25, current_x - group_gap + 0.25)
ax.set_ylabel(f'Average Score  (out of {NQ})')
ax.set_title('Average Score by Class and Period')
ax.set_ylim(0, NQ * 1.3); ax.grid(axis='y')
ax.spines['left'].set_linewidth(0.8); ax.spines['bottom'].set_linewidth(0.8)

ax2 = axes[1]
from scipy.ndimage import gaussian_filter1d as _gf1d

_sigma = max(0.7, NQ / 14)
for cls in classes:
    cls_df  = df[df['Class'] == cls]
    x_pts   = np.array(range(NQ + 1))
    pcts    = np.array([(cls_df['MC_Score'] == s).sum() / len(cls_df) for s in x_pts])
    color   = CLASS_COLORS.get(cls, C['blue'])
    y_smooth = np.clip(_gf1d(pcts.astype(float), sigma=_sigma), 0, None)
    ax2.plot(x_pts, y_smooth, '-', color=color, linewidth=2.5, alpha=0.9,
             label=f'{cls}  (n={len(cls_df)})')

ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax2.set_xticks(range(0, NQ + 1))
ax2.set_xlabel(f'Score  (out of {NQ})'); ax2.set_ylabel('Share of Class')
ax2.set_title('Score Distribution by Class')
ax2.legend(loc='upper left'); ax2.grid(axis='y')
ax2.spines['left'].set_linewidth(0.8); ax2.spines['bottom'].set_linewidth(0.8)

plt.tight_layout()
plt.show()

## 3.  Which Questions Were Hardest?
*Hover any cell for the full question and correct answer.*

In [ ]:
overall_p     = [df[f'Q{i}'].mean() for i in range(1, NQ + 1)]
order         = list(np.argsort(overall_p))
q_nums_sorted = [order[j] + 1 for j in range(NQ)]
col_labels    = [f'Q{q_nums_sorted[j]}' for j in range(NQ)]
full_q        = [str(q['Question'].tolist()[order[j]]).strip() for j in range(NQ)]
answers       = [str(q['Full_Answer'].tolist()[order[j]]).strip() for j in range(NQ)]
std_labels    = [std_name(q['Content_Std'].tolist()[order[j]], 35) for j in range(NQ)]

row_names = ['Overall'] + sorted(df['Class'].unique().tolist())
z_data, text_data, hover_data = [], [], []
for row_name in row_names:
    fdf   = df if row_name == 'Overall' else df[df['Class'] == row_name]
    n     = len(fdf)
    row_p = [fdf[f'Q{q_nums_sorted[j]}'].mean() for j in range(NQ)]
    z_data.append(row_p)
    text_data.append([f'{p:.0%}' for p in row_p])
    hover_data.append([
        f'<b>Q{q_nums_sorted[j]}: {std_labels[j]}</b><br>'
        f'{full_q[j]}<br><br>'
        f'\u2713 <b>Correct:</b> {answers[j]}<br>'
        f'{row_name} (n={n}): {row_p[j]:.0%} correct'
        for j in range(NQ)
    ])

fig = go.Figure(go.Heatmap(
    z=z_data, x=col_labels, y=row_names,
    colorscale=[[0, C['red']], [0.5, C['orange']], [1.0, C['teal']]],
    zmin=0, zmax=1,
    text=text_data, texttemplate='%{text}',
    textfont=dict(size=11, color='white'),
    hoverinfo='text', hovertext=hover_data,
    showscale=True,
    colorbar=dict(title='% Correct', tickformat='.0%',
                  tickvals=[0, 0.5, 0.7, 1.0],
                  ticktext=['0%', '50%', '70%', '100%']),
))
nat_order = list(range(NQ))
col_nat   = [f'Q{j+1}' for j in nat_order]
z_nat, txt_nat, hov_nat = [], [], []
for ri, row_name in enumerate(row_names):
    z_nat.append([z_data[ri][order.index(j)] for j in nat_order])
    txt_nat.append([f'{z_nat[-1][j]:.0%}' for j in range(NQ)])
    hov_nat.append([
        f'<b>Q{j+1}: {std_name(q["Content_Std"].tolist()[j], 35)}</b><br>'
        f'{str(q["Question"].tolist()[j]).strip()}<br><br>'
        f'\u2713 <b>Correct:</b> {str(q["Full_Answer"].tolist()[j]).strip()}<br>'
        f'{row_name}: {z_nat[-1][j]:.0%}'
        for j in nat_order
    ])

sort_btns = [
    dict(label='Hardest \u2192 Easiest', method='update',
         args=[{'x': [col_labels], 'z': [z_data], 'text': [text_data], 'hovertext': [hover_data]}]),
    dict(label='Question Order', method='update',
         args=[{'x': [col_nat],   'z': [z_nat],  'text': [txt_nat],   'hovertext': [hov_nat]}]),
]

fig.update_layout(**PLOTLY_BASE,
    title='Which Questions Were Hardest?',
    height=max(240, 80 + 70 * len(row_names)),
    xaxis=dict(side='top', title='', tickfont=dict(size=11), showgrid=False, ticklen=8),
    yaxis=dict(autorange='reversed', tickfont=dict(size=12), showgrid=False, ticklen=8),
    updatemenus=[dict(
        type='buttons', buttons=sort_btns, direction='right',
        showactive=True, x=0, xanchor='left', y=-0.1, yanchor='top',
        bgcolor='white', bordercolor='#CCCCCC', font=dict(size=10),
    )],
)
fig.add_shape(type='line', xref='paper', yref='y',
              x0=0, x1=1, y0=0.5, y1=0.5,
              line=dict(color='white', width=10), layer='above')
fig.update_layout(margin=dict(l=95, r=20, t=70, b=60))
fig.show()

## 4.  What Did Students Select?
*Select a question from the dropdown. Hover over each bar to see the full answer text.*

In [ ]:
has_opts = 'Opt1' in q.columns
fig = go.Figure()
q_text_map = {int(r['Q_Num']): str(r['Question']).strip() for _, r in q.iterrows()}

for i in range(NQ):
    q_num       = i + 1
    qrow        = q.loc[q['Q_Num'] == q_num].iloc[0]
    ca          = int(qrow['Answer'])
    opts        = ([str(qrow[f'Opt{j}']).strip() for j in range(1, 5)]
                   if has_opts else [str(j) for j in range(1, 5)])
    answered    = df[f'Q{q_num}_ans'].dropna().astype(int)
    total       = len(answered)
    counts      = answered.value_counts().reindex([1, 2, 3, 4], fill_value=0)
    pcts        = [counts[j] / total if total > 0 else 0 for j in range(1, 5)]

    fig.add_trace(go.Bar(
        x=[1, 2, 3, 4], y=pcts,
        marker_color=[C['teal'] if j == ca else C['lgray'] for j in range(1, 5)],
        marker_line_color='white', marker_line_width=1.2,
        text=[f'{p:.0%}' for p in pcts], textposition='outside',
        hovertext=[
            f'<b>Choice {j}: {opts[j-1]}</b><br>{pcts[j-1]:.0%} selected<br>'
            + ('\u2713 <b>Correct answer</b>' if j == ca else '\u2717 Incorrect')
            for j in range(1, 5)],
        hoverinfo='text', visible=(q_num == 1), showlegend=False, cliponaxis=False,
    ))

buttons = []
for q_num in range(1, NQ + 1):
    n_blank = df[f'Q{q_num}_ans'].isna().sum()
    blank_s = f'  ({n_blank} blank)' if n_blank else ''
    short   = q_text_map[q_num][:75] + ('\u2026' if len(q_text_map[q_num]) > 75 else '')
    buttons.append(dict(
        label=f'Q{q_num}',
        method='update',
        args=[{'visible': [i == (q_num - 1) for i in range(NQ)]},
              {'annotations[0].text': f'{short}{blank_s}'}],
    ))

short1 = q_text_map[1][:90] + ('\u2026' if len(q_text_map[1]) > 90 else '')
fig.update_layout(**PLOTLY_BASE,
    height=480,
    xaxis=dict(tickvals=[1, 2, 3, 4], title='', showgrid=False, zeroline=False),
    yaxis=dict(tickformat='.0%', title='% of Students', range=[0, 1.18],
               showgrid=True, gridcolor='#EEEEEE', zeroline=False),
    annotations=[dict(
        text=short1,
        x=0, y=-0.18, xref='paper', yref='paper',
        xanchor='left', yanchor='top',
        showarrow=False,
        font=dict(size=12, color=C['navy']),
    )],
    updatemenus=[dict(
        type='buttons',
        buttons=buttons,
        direction='right',
        showactive=True,
        x=0, xanchor='left',
        y=1.06, yanchor='bottom',
        bgcolor='white',
        bordercolor='#CCCCCC',
        font=dict(size=10),
        pad=dict(r=4, t=4, b=4),
    )],
)
fig.update_layout(margin=dict(l=0, r=20, t=55, b=100))
fig.show()

## 6.  How Are Different Student Groups Doing?

In [ ]:
masks = {
    'General Ed':       (~df['Is_ELL']) & (~df['Is_IEP']),
    'IEP only':          (~df['Is_ELL']) & df['Is_IEP'],
    'ELL only':          df['Is_ELL']  & (~df['Is_IEP']),
    'Both ELL & IEP':    df['Is_ELL']  & df['Is_IEP'],
}
groups_map  = {name: mask for name, mask in masks.items() if mask.sum() > 0}
grp_palette = [C['blue'], C['teal'], C['orange'], C['red']]
grp_names   = list(groups_map.keys())
grp_means   = [df[m]['MC_Score'].mean() for m in groups_map.values()]
grp_ns      = [int(m.sum())             for m in groups_map.values()]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(range(len(grp_names)), grp_means,
              color=grp_palette[:len(grp_names)], width=0.5, edgecolor='white')
for bar, mean, n in zip(bars, grp_means, grp_ns):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f'{mean:.1f}\n(n={n})', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(df['MC_Score'].mean(), color=C['navy'], linestyle=':', linewidth=1.5, alpha=0.6)
ax.text(len(grp_names) - 0.48, df['MC_Score'].mean() + 0.1,
        f'Overall: {df["MC_Score"].mean():.1f}', fontsize=9, color=C['navy'], alpha=0.7)
ax.set_xticks(range(len(grp_names))); ax.set_xticklabels(grp_names, fontsize=11)
ax.set_ylabel(f'Average Score  (out of {NQ})')
ax.set_title('Average Score by Student Group')
ax.set_ylim(0, NQ * 1.3); ax.grid(axis='y')
plt.tight_layout()
plt.show()

# Subgroup notes
print()
for flag, name in [('Is_ELL','ELL'), ('Is_IEP','IEP'), (None,'Both ELL & IEP')]:
    mask = (df['Is_ELL'] & df['Is_IEP']) if flag is None else df[flag]
    n = int(mask.sum())
    if n > 0:
        avg_g = df[mask]['MC_Score'].mean()
        avg_o = df[~mask]['MC_Score'].mean()
        gap   = avg_o - avg_g
        note  = f'{gap:+.1f} point gap' if abs(gap) >= 0.5 else 'minimal gap'
        print(f'  {name} (n={n}): avg {avg_g:.1f}/{NQ}  vs  {avg_o:.1f}/{NQ} for other students  \u2014 {note}')

## 7.  Key Takeaways

In [ ]:
from collections import defaultdict
std_groups = defaultdict(list)
for _, row in q.iterrows():
    qn   = int(row['Q_Num'])
    raw  = row.get('Content_Std', '')
    code = '' if (raw is None or str(raw).strip().lower() in ('nan', 'none', ''))            else str(raw).strip()
    std_groups[code or f'Q{qn}'].append(qn)

std_items = []
for code, qns in std_groups.items():
    avg_p = sum(df[f'Q{qn}'].mean() for qn in qns) / len(qns)
    label = std_name(code, 50) if code and not code.startswith('Q') else ''
    q_label = ', '.join(f'Q{n}' for n in sorted(qns))
    std_items.append((q_label, avg_p, label))

reteach = sorted([(ql,p,lbl) for ql,p,lbl in std_items if p < 0.50],      key=lambda x:x[1])
review  = sorted([(ql,p,lbl) for ql,p,lbl in std_items if 0.50<=p<0.70],  key=lambda x:x[1])
strong  = sorted([(ql,p,lbl) for ql,p,lbl in std_items if p >= 0.70],     key=lambda x:-x[1])

sections = [
    ('RETEACH',  'Below 50% correct',  reteach, C['red'],    '#fdf0f1'),
    ('REVIEW',   '50 \u2013 70% correct', review, C['orange'], '#fef8f0'),
    ('STRONG',   'Above 70% correct',  strong,  C['teal'],   '#f0fbf8'),
]
max_items = max(len(s[2]) for s in sections) or 1
fig_h = max(3.8, 1.6 + max_items * 0.62)

fig, panel_axes = plt.subplots(1, 3, figsize=(15, fig_h))
for ax, (title, subtitle, items, color, bg) in zip(panel_axes, sections):
    ax.set_facecolor(bg)
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_linewidth(2.5); sp.set_edgecolor(color)
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, 0.97, title, transform=ax.transAxes,
            ha='center', va='top', fontsize=14, fontweight='bold', color=color)
    ax.text(0.5, 0.88, subtitle, transform=ax.transAxes,
            ha='center', va='top', fontsize=10.5, color=C['gray'], style='italic')
    ax.plot([0.05, 0.95], [0.83, 0.83], transform=ax.transAxes,
            color=color, linewidth=0.9, alpha=0.45)
    if not items:
        ax.text(0.5, 0.55, 'None', transform=ax.transAxes,
                ha='center', va='center', fontsize=13, color=C['gray'], style='italic')
    else:
        for j, (ql, p, lbl) in enumerate(items):
            y = 0.78 - j * 0.10
            if y < 0.04: break
            ax.text(0.06, y, ql, transform=ax.transAxes,
                    ha='left', va='top', fontsize=11, fontweight='bold', color=color)
            ax.text(0.38, y, lbl, transform=ax.transAxes,
                    ha='left', va='top', fontsize=11, color=C['navy'])
            ax.text(0.97, y, f'{p:.0%}', transform=ax.transAxes,
                    ha='right', va='top', fontsize=12, fontweight='bold', color=color)

fig.suptitle(
    f'Key Takeaways  \u2014  avg: {df["MC_Score"].mean():.1f}/{NQ}  '
    f'({df["MC_Pct"].mean():.0%})  \u2014  '
    f'{(df["MC_Score"] >= round(NQ*0.67)).mean():.0%} scored {round(NQ*0.67)}+',
    fontsize=11, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

---
## Appendix — Technical Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
class_list   = sorted(df['Class'].unique())
class_colors = [CLASS_COLORS.get(c, C['blue']) for c in class_list]
data_cls     = [df[df['Class']==c]['MC_Score'].values for c in class_list]

bp = ax.boxplot(data_cls, patch_artist=True, labels=class_list, widths=0.45,
                medianprops={'color':'white','linewidth':2.5})
for patch, color in zip(bp['boxes'], class_colors):
    patch.set_facecolor(color); patch.set_alpha(0.65)
for j, (data, color) in enumerate(zip(data_cls, class_colors)):
    jitter = np.random.default_rng(j).normal(j+1, 0.07, len(data))
    ax.scatter(jitter, data, alpha=0.3, s=20, color=color, zorder=3)
ax.axhline(df['MC_Score'].mean(), color=C['navy'], linestyle='--', linewidth=1.2, alpha=0.6,
           label=f'Overall mean: {df["MC_Score"].mean():.1f}')
ax.set_ylabel(f'MC Score  (out of {NQ})'); ax.set_title('Score Distribution by Class')
ax.legend(fontsize=9.5); ax.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
pb_vals = [stats.pointbiserialr(df[f'Q{i}'], df['MC_Score'])[0] for i in range(1, NQ + 1)]
pb_labels = [f"Q{i}: {std_name(q.loc[q['Q_Num']==i,'Content_Std'].values[0], max_len=30)}"
             for i in range(1, NQ + 1)]
order_pb   = np.argsort(pb_vals)
pb_sorted  = [pb_vals[i]  for i in order_pb]
lbl_sorted = [pb_labels[i] for i in order_pb]
col_sorted = ['#E84855' if pb < 0.20 else '#F4A261' if pb < 0.30 else '#44BBA4' for pb in pb_sorted]

fig, ax = plt.subplots(figsize=(12, max(4, NQ * 0.45)))
bars = ax.barh(range(NQ), pb_sorted, color=col_sorted, height=0.6, edgecolor='white')
for bar, pb in zip(bars, pb_sorted):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
            f'{pb:.2f}', va='center', fontsize=10, fontweight='bold', color=C['navy'])
ax.set_yticks(range(NQ)); ax.set_yticklabels(lbl_sorted, fontsize=10)
ax.set_xlabel('Point-Biserial Correlation')
ax.set_title('Item Discrimination \u2014 Point-Biserial Correlation')
ax.grid(axis='x'); ax.spines['left'].set_visible(False); ax.tick_params(left=False)
ax.set_xlim(0, max(pb_sorted) + 0.12)
ax.legend(handles=[
    mpatches.Patch(color=C['red'],    label='< 0.20 \u2014 poor'),
    mpatches.Patch(color=C['orange'], label='0.20\u20130.30 \u2014 acceptable'),
    mpatches.Patch(color=C['teal'],   label='> 0.30 \u2014 good'),
], loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
p_vals_all  = [df[f'Q{i}'].mean() for i in range(1, NQ + 1)]
pb_vals_all = [stats.pointbiserialr(df[f'Q{i}'], df['MC_Score'])[0] for i in range(1, NQ + 1)]
q_topics    = [std_name(q.loc[q['Q_Num']==i,'Content_Std'].values[0], 35) for i in range(1, NQ + 1)]
q_texts_all = [str(q.loc[q['Q_Num']==i,'Question'].values[0]).strip()[:80] for i in range(1, NQ + 1)]

hover_txts = [
    f'<b>Q{i+1}: {q_topics[i]}</b><br>{q_texts_all[i]}\u2026<br><br>'
    f'Difficulty: {p_vals_all[i]:.0%} correct<br>Discrimination (PB): {pb_vals_all[i]:.2f}'
    for i in range(NQ)]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=p_vals_all, y=pb_vals_all, mode='markers+text',
    text=[f'Q{i}' for i in range(1, NQ + 1)], textposition='top right',
    textfont=dict(size=11, color=C['navy']),
    marker=dict(color=[pct_color(p) for p in p_vals_all], size=14,
                line=dict(color='white', width=1.5)),
    hovertext=hover_txts, hoverinfo='text', showlegend=False,
))
for label, color in [('Hard (<50%)', C['red']), ('Medium (50\u201370%)', C['orange']), ('Easy (>70%)', C['teal'])]:
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
        marker=dict(color=color, size=10), name=label, showlegend=True))
fig.add_hline(y=0.20, line_dash='dash', line_color=C['red'], opacity=0.5,
              annotation_text='PB = 0.20 (min)', annotation_font=dict(size=10, color=C['red']))
fig.add_hline(y=0.30, line_dash='dash', line_color=C['orange'], opacity=0.5,
              annotation_text='PB = 0.30', annotation_font=dict(size=10, color=C['orange']))
fig.add_vline(x=0.50, line_dash='dot', line_color=C['gray'], opacity=0.35)
fig.update_layout(**PLOTLY_BASE,
    title='Difficulty vs. Discrimination \u2014 hover any dot for details',
    height=480,
    legend=dict(orientation='h', y=-0.16, x=0.5, xanchor='center', font=dict(size=11)),
    xaxis=dict(tickformat='.0%', title='Difficulty  (% correct)',
               showgrid=True, gridcolor='#EEEEEE', zeroline=False, range=[0, 1.05]),
    yaxis=dict(title='Discrimination  (point-biserial)',
               showgrid=True, gridcolor='#EEEEEE', zeroline=False),
)
fig.show()